# 02 · Grover 搜索：用干涉放大正确答案

对应讲义实验 A · 约 30 分钟 · 前置：01 编程基础

在 $N = 2^n$ 个候选中找出唯一的目标，经典算法平均要查 $N/2$ 次，Grover 算法约 $\frac{\pi}{4}\sqrt{N}$ 次。它的核心是**干涉**：每一轮把目标的振幅放大一点。

**学完本节，你将能够**
- 用多控 Z 门实现 Oracle（翻转目标的符号）与扩散算子；
- 解释为什么迭代次数不是越多越好；
- 把算法推广到多个目标。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mindquantum import Circuit, UN, H, X, Z, Simulator
from mqlab import check

## 1. Oracle：只改相位，不改概率

多控 Z（`Z.on(n - 1, list(range(n - 1)))`）只给 $|1\cdots1\rangle$ 乘上 −1。要标记任意目标，先用 X 把目标中为 0 的位翻成 1，作用多控 Z，再翻回来。

In [ ]:
def oracle(target):              # |target⟩ → −|target⟩
    n = len(target)
    flips = Circuit()
    for i, b in enumerate(reversed(target)):
        if b == '0':
            flips += X.on(i)     # 0 位先翻成 1
    mcz = Z.on(n - 1, list(range(n - 1)))
    return flips + mcz + flips

先预测：对均匀叠加态作用 `oracle('101')` 之后，测到 101 的概率会变吗？

In [ ]:
sim = Simulator('mqvector', 3)
sim.apply_circuit(UN(H, 3) + oracle('101'))
amp = sim.get_qs().real
print('振幅：', amp.round(3))
print('概率：', (amp ** 2).round(3))

只有下标 5（即 `101`）的振幅变成了负数，概率完全没变：Oracle 把答案藏进了相位。

## 2. 扩散算子：把相位差变成振幅差

扩散算子关于均匀叠加态做反射：先用 H、X 把均匀叠加态变成 $|1\cdots1\rangle$，同样加负号，再变回来。效果相当于“关于平均值翻转”，负的振幅会变成大的正振幅。

In [ ]:
def diffusion(n):                # 关于均匀叠加态反射
    c = UN(H, n) + UN(X, n)
    c += Z.on(n - 1, list(range(n - 1)))   # 多控 Z
    return c + UN(X, n) + UN(H, n)

def grover(target, k):
    n = len(target)
    circ = UN(H, n)              # 均匀叠加
    for _ in range(k):           # k 次 Grover 迭代
        circ += oracle(target)
        circ += diffusion(n)
    return circ

grover('101', 1).svg()

## 3. 运行：先看理想概率，再采样

In [ ]:
circ = grover('101', k=2)
sim = Simulator('mqvector', circ.n_qubits)
sim.apply_circuit(circ)
probs = np.abs(sim.get_qs()) ** 2
print(probs.round(3))            # 下标 5 = '101'

In [ ]:
circ.measure_all()
sim.reset()                      # 先复位，再采样
res = sim.sampling(circ, shots=1000, seed=7)
print(res.data)

## 4. 迭代不是越多越好

成功率为 $\sin^2((2k+1)\theta)$，其中 $\sin\theta = 1/\sqrt{N}$。每一轮把态矢量朝目标转过 $2\theta$，转过头，概率就会下降。

In [ ]:
theta = np.arcsin(1 / np.sqrt(8))
p_sim = []
for k in range(7):
    s = Simulator('mqvector', 3)
    s.apply_circuit(grover('101', k))
    p_sim.append(abs(s.get_qs()[5]) ** 2)

kk = np.linspace(0, 6, 200)
plt.figure(figsize=(6, 3))
plt.plot(kk, np.sin((2 * kk + 1) * theta) ** 2, color='0.75', label='theory')
plt.plot(range(7), p_sim, 'o', color='#c0392b', label='simulator')
plt.xlabel('iterations k')
plt.ylabel('P(101)')
plt.legend()
plt.show()

### 练习 1 · 最优迭代次数

4 个比特、目标 `'0110'`，最优迭代次数是多少？先用公式 $k \approx \frac{\pi}{4\theta} - \frac{1}{2}$ 算出来，再用 `grover` 运行验证。

In [ ]:
# 练习 1：算出 k，再运行验证
k = ...
check.grover_iterations(k)

### 练习 2 · 多个目标

补全 `oracle_multi(targets)`：翻转每个目标态的符号，其余基态不变。提示：多个单目标 Oracle 可以依次相加。

In [ ]:
def oracle_multi(targets):
    circ = Circuit()
    # 练习 2
    ...
    return circ

check.oracle_multi(oracle_multi)

**挑战**：两个目标 `'101'` 和 `'011'`，只迭代 1 次，成功率是多少？为什么恰好如此？（提示：M 个目标时 $\sin\theta = \sqrt{M/N}$。）

In [ ]:
circ = UN(H, 3) + oracle_multi(['101', '011']) + diffusion(3)
s = Simulator('mqvector', 3)
s.apply_circuit(circ)
p = np.abs(s.get_qs()) ** 2
print('P(101) + P(011) =', round(p[5] + p[3], 4))

## 现场改一改：先预测，再运行

在上面的代码上改三处，每一处先写下预测：

1. `grover('101', k=4)`：测到 101 的概率会怎样？　A 接近 100%　B 仍约 94.5%　C 接近 0
2. `grover('011', k=2)`：Oracle 会在哪个比特上加 X？　A q0　B q1　C q2
3. 两个目标 `'101'`、`'011'`，只迭代 1 次：成功率是多少？　A 50%　B 78.1%　C 100%（就是上面的挑战题）

In [ ]:
s = Simulator('mqvector', 3)
s.apply_circuit(grover('101', k=4))
print('k = 4：P(101) =', round(abs(s.get_qs()[5]) ** 2, 3))
print("目标 '011' 的 Oracle：", [str(g) for g in oracle('011')])

<details><summary>答案</summary>

1. C：只剩 1.2%。每轮朝目标转过 2θ ≈ 41.4°，4 轮后已转到 186.3°，转过头了（见第 4 节的曲线）。
2. C：`'011'` 按 q2 q1 q0 书写，为 0 的是 q2，所以只在 q2 上加 X；k = 2 时测到 011 的概率同样是 94.5%。
3. C：$\sin\theta = \sqrt{2/8} = 1/2$，即 θ = 30°；初态与“非目标”方向夹角 30°，一次迭代转过 60°，恰好到达 90°，成功率 100%。
</details>

## 小结

- Oracle 只改相位；扩散算子把相位差变成振幅差。两步交替，目标振幅每轮约增大 $2/\sqrt{N}$。
- 最优迭代次数约为 $\frac{\pi}{4}\sqrt{N}$：查询次数从 $N/2$ 降到 $\sqrt{N}$ 量级，是**二次加速**。
- 在真机上，多控 Z 要分解成许多两比特门，线路很深——Grover 算法要在容错硬件上才有实际意义。